# Week 72, Day 3: Property-based testing

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w072d3.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


These are the files this page's programs use.

In [ ]:
# Run this cell first: it creates the files used below.
with open('props.py', 'w', encoding='utf-8') as f:
    f.write('"""A tiny property-testing library in the style of Hypothesis (for the browser)."""\nimport random\n\n\nclass Strategy:\n    def __init__(self, draw, shrink):\n        self.draw, self.shrink = draw, shrink\n\n\ndef integers(min_value=-1000, max_value=1000):\n    def draw(rng):\n        if rng.random() < 0.2:\n            return rng.choice([min_value, max_value, max(min_value, min(0, max_value))])\n        return rng.randint(min_value, max_value)\n\n    def shrink(n):\n        target = max(min_value, min(0, max_value))\n        if n != target:\n            yield target\n            half = (n + target) // 2 if n > target else -((target - n) // 2) + target\n            if half not in (n, target):\n                yield half\n            yield n - 1 if n > target else n + 1\n    return Strategy(draw, shrink)\n\n\ndef lists(elements, min_size=0, max_size=10):\n    def draw(rng):\n        size = min_size if rng.random() < 0.1 else rng.randint(min_size, max_size)\n        return [elements.draw(rng) for _ in range(size)]\n\n    def shrink(xs):\n        if len(xs) > min_size:\n            yield xs[:min_size]\n            for i in range(len(xs)):\n                yield xs[:i] + xs[i + 1:]\n        for i, x in enumerate(xs):\n            for smaller in elements.shrink(x):\n                yield xs[:i] + [smaller] + xs[i + 1:]\n    return Strategy(draw, shrink)\n\n\ndef text(alphabet="abc xyzABC019!", max_size=10):\n    chars = lists(sampled_from(list(alphabet)), max_size=max_size)\n    return Strategy(lambda rng: "".join(chars.draw(rng)),\n                    lambda s: ("".join(c) for c in chars.shrink(list(s))))\n\n\ndef sampled_from(options):\n    options = list(options)\n\n    def shrink(x):\n        i = options.index(x)\n        if i > 0:\n            yield options[0]\n    return Strategy(lambda rng: rng.choice(options), shrink)\n\n\ndef _fails(test, args):\n    try:\n        test(*args)\n    except AssertionError:\n        return True\n    return False\n\n\ndef _minimise(test, strategies, args):\n    improved = True\n    while improved:\n        improved = False\n        for i, strategy in enumerate(strategies):\n            for candidate in strategy.shrink(args[i]):\n                trial = args[:i] + [candidate] + args[i + 1:]\n                if _fails(test, trial):\n                    args, improved = trial, True\n                    break\n            if improved:\n                break\n    return args\n\n\ndef given(*strategies, runs=200, seed=0):\n    """Run the decorated test with many generated arguments; on failure, shrink the example."""\n    def decorate(test):\n        def wrapper():\n            rng = random.Random(seed)\n            for _ in range(runs):\n                args = [s.draw(rng) for s in strategies]\n                if _fails(test, args):\n                    smallest = _minimise(test, strategies, args)\n                    shown = ", ".join(repr(a) for a in smallest)\n                    raise AssertionError(f"Falsifying example: {test.__name__}({shown})")\n        wrapper.__name__ = test.__name__\n        return wrapper\n    return decorate\n')

**Predict before you run:** what will this print?



In [ ]:
import random

def middle(xs):
    """The middle item of a list (the lower middle for an even length)."""
    return sorted(xs)[len(xs) // 2]

rng = random.Random(1)
found = {}
for trial in range(1000):
    xs = [rng.randint(0, 9) for _ in range(rng.randint(0, 4))]
    try:
        m = middle(xs)
        below = sum(1 for x in xs if x < m)
        assert below <= (len(xs) - 1) // 2, "not the lower middle"
    except (AssertionError, IndexError) as e:
        found.setdefault(type(e).__name__, (trial, xs))
for kind, (trial, xs) in found.items():
    print(kind, "first at trial", trial, "with", xs)

In [ ]:
import random
from collections import Counter

def my_sort(xs):
    return sorted(xs)

rng = random.Random(7)
for _ in range(300):
    xs = [rng.randint(-3, 3) for _ in range(rng.randint(0, 6))]
    out = my_sort(xs)
    assert all(a <= b for a, b in zip(out, out[1:])), xs        # in order
    assert Counter(out) == Counter(xs), xs                      # same items
    assert my_sort(out) == out, xs                              # idempotent
    shift = [x + 5 for x in xs]
    assert my_sort(shift) == [x + 5 for x in out], xs           # metamorphic
print("all four properties held for 300 random lists")

In [ ]:
from props import given, integers, lists, text

def run_lengths(s):
    out = []
    for ch in s:
        if out and out[-1][0] == ch:
            out[-1][1] += 1
        else:
            out.append([ch, 1])
    return [(ch, n) for ch, n in out]

@given(text(alphabet="ab"))
def test_lengths_add_up(s):
    assert sum(n for _, n in run_lengths(s)) == len(s)

@given(text(alphabet="ab"))
def test_rebuilds(s):
    assert "".join(ch * n for ch, n in run_lengths(s)) == s

@given(lists(integers(-5, 5), max_size=6))
def test_reverse_twice(xs):
    assert xs[::-1][::-1] == xs

for test in (test_lengths_add_up, test_rebuilds, test_reverse_twice):
    test()
    print(test.__name__, "passed 200 generated cases")

In [ ]:
from props import given, integers, lists

def has_no_duplicates(xs):
    return len(set(xs)) == len(xs)

@given(lists(integers(0, 1000)))
def test_unique(xs):
    assert has_no_duplicates(xs)

try:
    test_unique()
except AssertionError as e:
    print(e)

@given(integers(0, 10_000), integers(0, 10_000))
def test_small_sum(a, b):
    assert a + b < 1234

try:
    test_small_sum()
except AssertionError as e:
    print(e)

> **Key idea.** A property must hold for every valid input: round trips, oracles, invariants, metamorphic relations, idempotence, or simply not crashing. Avoid properties that repeat the code, and combine weak ones (in order and same items). Strategies generate values with a bias towards edge cases and know how to shrink them; given runs the test on many drawn inputs from a fixed seed. Shrinking keeps any smaller candidate that still fails until none does, giving a minimal example. Stateful tests compare an object with a simple model over random sequences of operations. Hypothesis does all of this for real.

### Exercise: A strategy of your own

Using `Strategy(draw, shrink)` from `props.py`, write `tuples(*strategies)`: a strategy whose `draw(rng)` returns a tuple with one value drawn from each strategy, in order, and whose `shrink(value)` yields candidates made by shrinking **one** position at a time with that position's strategy (all candidates for position 0 first, then position 1, and so on).

In [ ]:
from props import Strategy, integers, sampled_from

def tuples(*strategies):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_tuples():\n    \"drawing and shrinking\"\n    import random\n    pair = tuples(integers(0, 9), sampled_from(\"xyz\"))\n    rng = random.Random(3)\n    values = [pair.draw(rng) for _ in range(50)]\n    assert all(isinstance(v, tuple) and len(v) == 2 and 0 <= v[0] <= 9 and v[1] in \"xyz\" for v in values)\n    assert list(pair.shrink((6, \"z\"))) == [(0, \"z\"), (3, \"z\"), (5, \"z\"), (6, \"x\")]\n    assert list(pair.shrink((0, \"x\"))) == [] and tuples().draw(rng) == ()")

### Exercise: Round trips

`rle.py` has `encode(s)`, which turns a string into a list of `(character, count)` pairs for each run (`"aaab"` becomes `[("a", 3), ("b", 1)]`), and `decode(pairs)`. Write property tests with `@given` from `props.py` (a round trip, plus at least one more property or known answer) that pass on the correct code and catch every broken version.

In [ ]:
# Run this cell first: it creates the files used below.
with open('rle.py', 'w', encoding='utf-8') as f:
    f.write('def encode(s):\n    pairs = []\n    for ch in s:\n        if pairs and pairs[-1][0] == ch:\n            pairs[-1] = (ch, pairs[-1][1] + 1)\n        else:\n            pairs.append((ch, 1))\n    return pairs\n\ndef decode(pairs):\n    return "".join(ch * n for ch, n in pairs)\n')

In [ ]:
from props import given, text
from rle import encode, decode

@given(text())
def test_round_trip(s):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("GOOD = open(\"rle.py\").read()\nBROKEN = {\n    \"loses the last run\": GOOD.replace(\"    return pairs\", \"    return pairs[:-1]\"),\n    \"never merges\": GOOD.replace(\"if pairs and pairs[-1][0] == ch:\", \"if False:\"),\n    \"decodes one of each\": GOOD.replace(\"ch * n for ch, n in pairs\", \"ch for ch, n in pairs\"),\n    \"counts start at zero\": GOOD.replace(\"pairs.append((ch, 1))\", \"pairs.append((ch, 0))\").replace(\"ch * n for\", \"ch * (n + 1) for\"),\n}\ndef test_catches():\n    \"your properties pass on the correct code and catch all four broken versions\"\n    assert uses(\"given\"), \"use @given\"\n    results = against(source, \"rle\", {\"correct\": GOOD, **BROKEN})\n    write_file(\"rle.py\", GOOD)\n    assert results[\"correct\"] == [], f\"these fail on the correct code: {results['correct']}\"\n    missed = [k for k in BROKEN if not results[k]]\n    assert not missed, f\"not caught: {missed}\"")

### Exercise: Checking against an oracle

`intervals.py` has `merge(intervals)`, a fast function that merges overlapping or touching `(start, end)` intervals into a sorted list. Write `slow_merge(intervals)` in your test file as an **oracle** (mark every covered whole number in a set, then read off the runs; intervals here have small whole-number ends, with `start <= end`), and a `@given` property that the two agree. It must catch every broken version.

In [ ]:
# Run this cell first: it creates the files used below.
with open('intervals.py', 'w', encoding='utf-8') as f:
    f.write('def merge(intervals):\n    out = []\n    for start, end in sorted(intervals):\n        if out and start <= out[-1][1] + 1:\n            out[-1] = (out[-1][0], max(out[-1][1], end))\n        else:\n            out.append((start, end))\n    return out\n')

In [ ]:
from props import given, lists, integers, Strategy
from intervals import merge

def slow_merge(intervals):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("GOOD = open(\"intervals.py\").read()\nBROKEN = {\n    \"doesn't sort\": GOOD.replace(\"in sorted(intervals)\", \"in intervals\"),\n    \"touching not merged\": GOOD.replace(\"start <= out[-1][1] + 1\", \"start <= out[-1][1]\"),\n    \"keeps the later end\": GOOD.replace(\"max(out[-1][1], end)\", \"end\"),\n}\ndef test_catches():\n    \"an oracle catches all three broken versions\"\n    results = against(source, \"intervals\", {\"correct\": GOOD, **BROKEN})\n    write_file(\"intervals.py\", GOOD)\n    assert results[\"correct\"] == [], f\"these fail on the correct code: {results['correct']}\"\n    missed = [k for k in BROKEN if not results[k]]\n    assert not missed, f\"not caught: {missed}\"")

### Exercise: Relations, not answers

`stats.py` has `mean(xs)` and `spread(xs)` (the largest minus the smallest). Write **metamorphic** properties: adding a constant to every number adds it to the mean and leaves the spread the same; doubling every number doubles both; reversing the list changes neither. Use `@given`, integers and `pytest.approx` for the mean, and catch every broken version (the lists must not be empty).

In [ ]:
# Run this cell first: it creates the files used below.
with open('stats.py', 'w', encoding='utf-8') as f:
    f.write('def mean(xs):\n    return sum(xs) / len(xs)\n\ndef spread(xs):\n    return max(xs) - min(xs)\n')

In [ ]:
import pytest
from props import given, lists, integers
from stats import mean, spread

@given(lists(integers(-50, 50), min_size=1), integers(-10, 10))
def test_shift(xs, k):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("GOOD = open(\"stats.py\").read()\nBROKEN = {\n    \"mean of the first and last\": GOOD.replace(\"return sum(xs) / len(xs)\", \"return (xs[0] + xs[-1]) / 2\"),\n    \"spread from the ends of the list\": GOOD.replace(\"return max(xs) - min(xs)\", \"return abs(xs[-1] - xs[0])\"),\n    \"mean divides by one too many\": GOOD.replace(\"sum(xs) / len(xs)\", \"sum(xs) / (len(xs) + 1)\"),\n    \"spread is the largest\": GOOD.replace(\"return max(xs) - min(xs)\", \"return max(xs)\"),\n}\ndef test_catches():\n    \"metamorphic properties catch all four broken versions\"\n    results = against(source, \"stats\", {\"correct\": GOOD, **BROKEN})\n    write_file(\"stats.py\", GOOD)\n    assert results[\"correct\"] == [], f\"these fail on the correct code: {results['correct']}\"\n    missed = [k for k in BROKEN if not results[k]]\n    assert not missed, f\"not caught: {missed}\"")

### Exercise: Testing against a model

`lru.py` has `LRUCache(capacity)` with `get(key)` (returning `None` if absent, and marking the key as recently used) and `put(key, value)` (evicting the least recently used key when full). Write a property test that draws a **list of operations** (such as `("put", key, value)` and `("get", key)`, with keys from a small set so they repeat), applies each one both to the cache and to a **model** you write with a plain list or `OrderedDict`, and checks every `get` agrees. It must catch every broken version.

In [ ]:
# Run this cell first: it creates the files used below.
with open('lru.py', 'w', encoding='utf-8') as f:
    f.write('from collections import OrderedDict\n\nclass LRUCache:\n    def __init__(self, capacity):\n        self.capacity = capacity\n        self._data = OrderedDict()\n\n    def get(self, key):\n        if key not in self._data:\n            return None\n        self._data.move_to_end(key)\n        return self._data[key]\n\n    def put(self, key, value):\n        if key in self._data:\n            self._data.move_to_end(key)\n        self._data[key] = value\n        if len(self._data) > self.capacity:\n            self._data.popitem(last=False)\n')

In [ ]:
from props import given, lists, sampled_from, Strategy
from lru import LRUCache

def test_matches_model():
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("GOOD = open(\"lru.py\").read()\nBROKEN = {\n    \"get doesn't refresh\": GOOD.replace(\"        self._data.move_to_end(key)\\n        return self._data[key]\", \"        return self._data[key]\"),\n    \"put doesn't refresh\": GOOD.replace(\"        if key in self._data:\\n            self._data.move_to_end(key)\\n\", \"\"),\n    \"evicts the newest\": GOOD.replace(\"popitem(last=False)\", \"popitem()\"),\n    \"one too many\": GOOD.replace(\"len(self._data) > self.capacity\", \"len(self._data) > self.capacity + 1\"),\n}\ndef test_catches():\n    \"a model-based test catches all four broken versions\"\n    results = against(source, \"lru\", {\"correct\": GOOD, **BROKEN})\n    write_file(\"lru.py\", GOOD)\n    assert results[\"correct\"] == [], f\"these fail on the correct code: {results['correct']}\"\n    missed = [k for k in BROKEN if not results[k]]\n    assert not missed, f\"not caught: {missed}\"")

### Exercise: Shrinking by hand

Write `shrink_text(s)`: a generator of smaller candidates for a string: first the empty string (if `s` isn't empty), then `s` with each single character removed (from the first position to the last), then `s` with each character that isn't `"a"` replaced by `"a"`. Then `minimise(value, fails, shrink)`: repeatedly take the **first** candidate from `shrink(value)` for which `fails(candidate)` is true, until none is; return the final value.

In [ ]:
def shrink_text(s):
    ___

def minimise(value, fails, shrink):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_shrinking():\n    \"candidates and a greedy search\"\n    assert list(shrink_text(\"xy\")) == [\"\", \"y\", \"x\", \"ay\", \"xa\"] and list(shrink_text(\"\")) == []\n    assert list(shrink_text(\"ab\")) == [\"\", \"b\", \"a\", \"aa\"]\n    assert minimise(\"hello world\", lambda s: \" \" in s, shrink_text) == \" \"\n    assert minimise(\"Zebra!\", lambda s: any(c.isupper() for c in s) and len(s) >= 2, shrink_text) == \"Za\"\n    assert minimise(1000, lambda n: n >= 37, lambda n: (c for c in (0, n // 2, n - 1) if c < n)) == 37")

### Bug hunt: A property that's too weak

The only property checks that the result is in order, so the broken `my_sort` (which throws everything away and returns an empty list) passes. Add the property that the result has the **same items**, so the broken version is caught.

In [ ]:
# Run this cell first: it creates the files used below.
with open('sorting.py', 'w', encoding='utf-8') as f:
    f.write('def my_sort(xs):\n    return sorted(xs)\n')

In [ ]:
from props import given, lists, integers
from sorting import my_sort

@given(lists(integers(-9, 9)))
def test_sorted(xs):
    out = my_sort(xs)
    assert all(a <= b for a, b in zip(out, out[1:]))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_fixed():\n    \"the empty result is caught\"\n    good = open(\"sorting.py\").read()\n    results = against(source, \"sorting\", {\"correct\": good, \"loses everything\": \"def my_sort(xs):\\n    return []\\n\", \"loses repeats\": \"def my_sort(xs):\\n    return sorted(set(xs))\\n\"})\n    write_file(\"sorting.py\", good)\n    assert results[\"correct\"] == [] and results[\"loses everything\"] and results[\"loses repeats\"], results")

### Bug hunt: A strategy that never tries the edge

`average` should return `0` for an empty list, and the broken version crashes instead. The strategy uses `min_size=1`, so the property never sees an empty list and the bug is missed. Let the lists be empty too, and keep the property true for every list.

In [ ]:
# Run this cell first: it creates the files used below.
with open('averages.py', 'w', encoding='utf-8') as f:
    f.write('def average(xs):\n    if not xs:\n        return 0\n    return sum(xs) / len(xs)\n')

In [ ]:
from props import given, lists, integers
from averages import average

@given(lists(integers(0, 10), min_size=1))
def test_average_in_range(xs):
    assert 0 <= average(xs) <= 10

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_fixed():\n    \"the empty list is tested\"\n    good = open(\"averages.py\").read()\n    broken = \"def average(xs):\\n    return sum(xs) / len(xs)\\n\"\n    results = against(source, \"averages\", {\"correct\": good, \"crashes when empty\": broken})\n    write_file(\"averages.py\", good)\n    assert results[\"correct\"] == [] and results[\"crashes when empty\"], results")

### Bug hunt: A shrinker that gives up early

`minimise` should keep shrinking until no candidate fails, but it returns as soon as it finds the **first** smaller failing value, so it reports `500` instead of the minimal `37`. Make it continue from each improvement so this prints `37`.

In [ ]:
def minimise(n, fails):
    while True:
        for candidate in (0, n // 2, n - 1):
            if candidate < n and fails(candidate):
                n = candidate
                return n
        else:
            return n

print(minimise(1000, lambda n: n >= 37))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"a full search\"\n    expect_output(\"37\")")

### Puzzle: A round-trip property

Arrange the lines of a property test that checks JSON round trips for lists of integers. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # the kind of property "sorted(sorted(xs)) == sorted(xs)" is: "idempotence" or "round trip"
q = ___   # True or False: "the output is in order" alone catches a sort that returns []
r = ___   # list(integers(0, 100).shrink(8)) with props.py
s = ___   # True or False: shrinking keeps a candidate only if the test still fails on it
t = ___   # the kind of property "mean([x + k for x in xs]) == mean(xs) + k" is: "oracle" or "metamorphic"

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == \"idempotence\" and q is False and r == [0, 4, 7], \"p, q and r\"\n    assert s is True and t == \"metamorphic\", \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a stateful tester

Write `check_machine(make_real, make_model, draw_op, apply, steps=20, runs=100, seed=0)`. For each run, draw a sequence of `steps` operations with `draw_op(rng)` (using one `random.Random(seed)` for the whole search), then replay it on a fresh real object and a fresh model: `apply(obj, op)` performs one operation and returns its result. If, at some step, the two results differ, or one raises and the other doesn't, the sequence **fails**. On the first failing sequence, shrink it: cut it off just after the step that failed, then repeatedly try removing each single operation (from the first to the last), keeping the first removal that still fails, until no removal does. Return the shrunk list of operations, or `None` if every run passed.

In [ ]:
import random

def check_machine(make_real, make_model, draw_op, apply, steps=20, runs=100, seed=0):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_machine():\n    \"finding and shrinking a sequence\"\n    class BuggyStack:\n        def __init__(self):\n            self.items = []\n        def push(self, x):\n            if len(self.items) < 3:          # silently drops the 4th push\n                self.items.append(x)\n        def pop(self):\n            return self.items.pop() if self.items else None\n    class ModelStack(BuggyStack):\n        def push(self, x):\n            self.items.append(x)\n    def draw_op(rng):\n        return (\"push\", rng.randint(0, 9)) if rng.random() < 0.6 else (\"pop\",)\n    def apply(obj, op):\n        return obj.push(op[1]) if op[0] == \"push\" else obj.pop()\n    found = check_machine(BuggyStack, ModelStack, draw_op, apply)\n    assert found is not None and [op[0] for op in found] == [\"push\", \"push\", \"push\", \"push\", \"pop\"], found\n    assert check_machine(ModelStack, ModelStack, draw_op, apply) is None\n    class Crashy(ModelStack):\n        def pop(self):\n            return self.items.pop()          # raises when empty\n    found = check_machine(Crashy, ModelStack, draw_op, apply)\n    assert found == [(\"pop\",)], found")

## Warm-up

Two bugs, found in seconds, that a handful of hand-written examples might never show: the empty list crashes, and for even lengths `len(xs) // 2` picks the **upper** middle, not the lower one the docstring promises. Random inputs, a rule and a loop are the whole idea. The rest of today is about doing it well.

## Properties instead of examples

A **property** is a statement that must be true for **every** valid input. The hard part is finding properties that are both true and strong. Patterns that work again and again:

- **Round trip**: undo what you did. `decode(encode(x)) == x`; `json.loads(json.dumps(d)) == d`; `parse(format(date)) == date`.
- **Oracle**: compare with a simple, obviously correct version: a clever fast function must agree with a slow brute-force one (Phase 4 is full of these pairs).
- **Invariant**: something that doesn't change. Sorting keeps the same items (`Counter(sorted(xs)) == Counter(xs)`); a transfer keeps the total money the same.
- **Metamorphic relation**: you don't know the answer, but you know how it changes when the input changes. Adding a constant to every number adds it to the mean; shuffling the input doesn't change a sum; searching for more words never returns more results.
- **Idempotence**: doing it twice is the same as doing it once: `normalise(normalise(s)) == normalise(s)`, `sorted(sorted(xs)) == sorted(xs)`.
- **It doesn't crash**: the weakest property, and still valuable for parsers: any input either parses or raises the documented error.

Two warnings. A property that **repeats the code's own logic** (`assert vat(p) == p * 1.2` when `vat` is `p * 1.2`) checks nothing. And a single property is often too weak: "the output is sorted" is satisfied by returning `[]`, which is why sorting needs **both** "in order" and "same items". Properties combine well with a few known-answer examples.

## Strategies and given

Generating inputs by hand in a loop gets repetitive, and the inputs are usually poor: uniformly random numbers almost never hit `0`, the boundaries or duplicates, which is where bugs live. A property-testing library separates **what** to generate from the test. This page has one, `props.py` (open it above): about a hundred lines in the style of **Hypothesis**, the library Python projects use for real.

- A **strategy** describes how to generate one kind of value, and how to make it smaller. `integers(min_value, max_value)`, `lists(elements, min_size, max_size)`, `text(alphabet, max_size)` and `sampled_from(options)` build them, and they compose: `lists(lists(integers(0, 9)))` is lists of lists of digits.
- **`@given(strategy, ...)`** turns a function with arguments into a test with none: it draws arguments from the strategies (200 times by default, from a fixed seed, so a failure always reproduces) and runs the function with them. pytest collects the result like any other test.
- Good strategies are **biased towards edge cases**. `integers` returns its minimum, maximum or zero a fifth of the time; `lists` is often empty or as short as allowed.

## Shrinking

When a property fails on a random input, the input is usually messy: a list of nine numbers in the hundreds, when the bug needs only two equal numbers. **Shrinking** searches for a smaller input that **still fails**, and reports that one instead:

1. Each strategy knows how to suggest **smaller candidates** for a value: an integer towards zero (zero itself, then halfway, then one step); a list by dropping items, and by shrinking each item.
2. The library tries the candidates in turn. As soon as one still fails, it becomes the current example and the search starts again from it.
3. When no candidate fails, the example is **minimal**: every simpler suggestion passes, so whatever's left is what the bug needs.

The result is the difference between "failed on `[731, -12, 731, 88, 5, ...]`" and "failed on `[0, 0]`": the second tells you the bug at once.

The second failure shrinks to two numbers adding up to exactly 1234: the **boundary**, found automatically.

## Stateful testing, and the real Hypothesis

Some objects have **state**: a cache, a bank account, a queue. Their bugs appear after a particular **sequence** of operations. Stateful testing generates random sequences of operations, applies each to the real object **and** to a simple **model** (often a dict or a list, obviously correct but slow), and checks they agree after every step. When they disagree, the sequence is shrunk like any other input, by dropping operations, until it's a short recipe for the bug.

On your own computer, use **Hypothesis** itself (`pip install hypothesis`). It works like `props.py`, with far better strategies and shrinking, a database that remembers failures between runs, and a full stateful-testing system:

```python
from hypothesis import given, strategies as st

@given(st.lists(st.integers()))
def test_sort_keeps_items(xs):
    assert sorted(sorted(xs)) == sorted(xs)

@given(st.text(), st.integers(min_value=0, max_value=25))
def test_cipher_round_trip(s, k):
    assert decode(encode(s, k), k) == s
```

`st.builds(Account, owner=st.text())` makes objects, `@example(...)` pins a known case, `assume(...)` discards inputs that don't fit, and `hypothesis.stateful.RuleBasedStateMachine` writes stateful tests as classes with `@rule` methods.

## Practice

## Quiz

## Daily challenge

## Before you go

Write one property for a function you've written in this course: a round trip, an oracle or a relation. Tomorrow: **coverage and mutation testing**, measuring how much of your code the tests run, and, more tellingly, how many deliberately planted bugs they catch.